In [31]:
#Import Packages
import sys, getopt, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliFeatureMap, RealAmplitudes
from qiskit.primitives import Sampler
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.circuit.library import QNNCircuit
from qiskit_machine_learning.utils.loss_functions import L2Loss  # Qiskit ML loss, or use PyTorch's
from qiskit.quantum_info import SparsePauliOp

In [32]:
#Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [33]:
root_folder = 'QNNC_hybrid'
### Globals
# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Fixed feature sizes
NUM_FEATURES = 3
NUM_QUBITS = NUM_FEATURES
NUM_TARGETS = 1

# Quantum circuit parameters
FEATURE_MAP_REPS_LIST = [1]
ANSATZ_REPS_LIST = [1]
ENTANGLEMENT_LIST = ['linear', 'full', 'circular']

# Training hyperparameters
LEARNING_RATE = 0.01
BATCH_SIZE = 30
NUM_EPOCHS = 100  # Adjust as needed

# K-fold cross-validation parameters
N_REPEATS = 1
TEST_SIZE = 1  # Leave-one-out cross-validation (LOOCV) is suggested since the sample size is too small

# Data conf
CLASSIFIER_THRESHOLD = 19


In [34]:
def get_qnn_torch_model(entangle, feature_map_reps, ansatz_reps):
    ### Feature Map, Ansatz, then QNN Constructor
    # a. Feature Map: Encodes NUM_FEATURES into NUM_QUBITS
    # ParameterVector for input features
    input_params = ParameterVector("x", NUM_FEATURES)

    feature_map_template = PauliFeatureMap(
        feature_dimension=NUM_FEATURES,  # This tells the template how many input parameters it structurally needs
        reps=feature_map_reps,
        entanglement=entangle
    )

    # Assign the *specific* input parameters from the vector to the template's parameter slots
    # This creates a new circuit instance containing parameters ONLY from input_params (size NUM_FEATURES)
    feature_map = feature_map_template.assign_parameters(input_params)
    print(f"Assigned feature map parameters: {feature_map.num_parameters}")

    # Create a template to find out how many parameters it needs structurally
    ansatz_template = RealAmplitudes(NUM_QUBITS, reps=ansatz_reps, entanglement=entangle)
    # ParameterVector for trainable weights - sized based on the template's structural parameters
    num_ansatz_params = ansatz_template.num_parameters  # This was correctly calculated as 12
    weight_params = ParameterVector("θ", num_ansatz_params)

    # Create the ansatz circuit instance by assigning the weight parameters to the template
    ansatz = ansatz_template.assign_parameters(weight_params)
    print(f"Assigned ansatz parameters: {ansatz.num_parameters}")

    # c. Combine into a full quantum circuit
    qc = QNNCircuit(
        feature_map=feature_map,
        ansatz=ansatz_template,
    )

    # example 5.2 from Qiskit guide on binary classification
    parity = lambda x: "{:b}".format(x).count("1") % 2
    output_shape = 2  # parity = 0, 1

    sampler = Sampler()

    qnn = SamplerQNN(
        circuit=qc,
        interpret=parity,
        output_shape=output_shape,
        sampler=sampler,
        sparse=False,
        input_gradients=False,  # Set to True if you need gradients w.r.t. inputs
    )

    # --- 4. TorchConnector ---
    # Wrap the QNN into a PyTorch module
    initial_weights = 0.01 * (2 * np.random.rand(qnn.num_weights) - 1)
    qnn_torch_model = TorchConnector(qnn, initial_weights=torch.tensor(initial_weights, dtype=torch.float32))

    return qnn_torch_model.to(device)


In [35]:
class HybridModel(nn.Module):
    def __init__(self, qnn_model):
        super().__init__()
        # Example: Add classical layers if needed
        # self.classical_pre = nn.Linear(NUM_FEATURES, NUM_FEATURES) # If you want to pre-process features
        self.qnn = qnn_model
        # Example: Add classical layers after the QNN
        # self.classical_post = nn.Linear(qnn_model.output_shape[0], NUM_TARGETS) # output_shape[0] is num_observables
        # If qnn_model.output_shape is (1,), then it's 1.

    def forward(self, x):
        # x = self.classical_pre(x) # If using classical_pre
        x = self.qnn(x)
        # x = self.classical_post(x) # If using classical_post
        return x


In [36]:
def prepare_dataset_k_fold(X, y, train_indices, test_indices):
    # Separate train/test split
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    # Separate element column from the actual features
    element_test = X_test_raw[:, 0]
    element_train = X_train_raw[:, 0]

    # Drop the element column (first column)
    X_train = X_train_raw[:, 1:]
    X_test = X_test_raw[:, 1:]

    full_X = np.vstack([X_train, X_test])

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(full_X)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, y_train, X_test_scaled, y_test, element_test, element_train


In [37]:
def get_arguments(argvs):
    _entangle = ''
    _feature_map_reps = ''
    _ansatz_reps = ''
    try:
        opts, args = getopt.getopt(argvs, "h:e:f:a:", ["entangle=", "feature_map_reps=", "ansatz_reps="])
    except getopt.GetoptError:
        print(root_folder + '.py -e <entangle> -f <feature_map_reps> -a <ansatz_reps>')
        sys.exit(2)
    for opt, arg in opts:
        if opt == '-h':
            print(root_folder + '.py -e <entangle> -f <feature_map_reps> -a <ansatz_reps>')
            sys.exit()
        elif opt in ("-e", "--entangle"):
            _entangle = arg
        elif opt in ("-f", "--feature_map_reps"):
            _feature_map_reps = int(arg)
        elif opt in ("-a", "--ansatz_reps"):
            _ansatz_reps = int(arg)
    return _entangle, _feature_map_reps, _ansatz_reps

In [38]:
date = '06_08_25_1'
if not os.path.exists(f'{root_folder}/result'):
    os.makedirs(f'{root_folder}/result')
if not os.path.exists(f'{root_folder}/logs'):
    os.makedirs(f'{root_folder}/logs')

In [39]:
dataset_name = "qml_training-validation-data.csv"
df = pd.read_csv(dataset_name)
display(df.head())
X = df[['Element', 'el_neg', 'B/GPa', 'Volume/A^3']].values
y = df['SFE/mJm^-3'].values
print(df.shape)

,Element,el_neg,B/GPa,Volume/A^3,SFE/mJm^-3
0,Be,1.57,130.0,8.09,23.48
1,Sc,1.36,57.0,25.00,16.16
2,Ti,1.54,110.0,17.60,24.44
3,Co,1.88,180.0,11.00,37.64
4,Zn,1.65,70.0,15.20,20.98


(21, 5)


In [40]:
# Group data by classifier threshold
for i in range(0, len(y)):
    if y[i] > CLASSIFIER_THRESHOLD:
        y[i] = 0
    else:
        y[i] = 1

# y_scaler = MinMaxScaler(feature_range=(-1, 1))
# y = y_scaler.fit_transform(y.reshape(-1, 1))

In [45]:
print('Total number of data: ', X.shape[0])
rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
print(rkf)

Total number of data:  21
RepeatedKFold(n_repeats=1, n_splits=21, random_state=None)


In [42]:
df = pd.DataFrame(columns=['entanglement', 'feature_map_reps', 'ansatz_reps',
                            'element test', 'actual test', 'predicted test',
                            'element train', 'actual train', 'predicted train',
                            ])

In [43]:
# Build output filename
if len(FEATURE_MAP_REPS_LIST) == 1:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST[0]
else:
    FEATURE_MAP_REPS_LIST_NAME = FEATURE_MAP_REPS_LIST
if len(ANSATZ_REPS_LIST) == 1:
    ANSATZ_REPS_LIST_NAME = ANSATZ_REPS_LIST[0]
else:
    ANSATZ_REPS_LIST_NAME = ANSATZ_REPS_LIST
if len(ENTANGLEMENT_LIST) == 1:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST[0]
else:
    ENTANGLEMENT_LIST_NAME = ENTANGLEMENT_LIST
file_name = f'{root_folder}/result/FMR_{FEATURE_MAP_REPS_LIST_NAME}_AR_{ANSATZ_REPS_LIST_NAME}_E_{ENTANGLEMENT_LIST_NAME}_{date}.csv'
print(file_name)

QNNC_hybrid/result/FMR_1_AR_1_E_['linear', 'full', 'circular']_06_08_25_1.csv


In [44]:
i = 0
LOSS = nn.CrossEntropyLoss()  # use torch.long
    # LOSS = nn.MSELoss() # haven't tried
    # LOSS = nn.BCELoss() # use torch.float

print("\n--- Start K-Fold Loop ---")

for train_indices, test_indices in rkf.split(X):
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, y_train, X_test, y_test, element_test, element_train = prepare_dataset_k_fold(X, y, train_indices, test_indices)

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    # Create DataLoaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Training data shape: X_train_t: {X_train_t.shape}, y_train_t: {y_train_t.shape}")
    print(f"Testing data shape: X_test_t: {X_test_t.shape}, y_test_t: {y_test_t.shape}")

    # For binary classification (0 or 1 target):
    # You might scale the QNN output (e.g., (output + 1) / 2 to get [0,1]) and then use nn.BCELoss()
    # Or use nn.BCEWithLogitsLoss() if your QNN output is treated as logits (less common for direct EstimatorQNN output).

    for entanglement in ENTANGLEMENT_LIST:
        for feature_map_reps in FEATURE_MAP_REPS_LIST:
            for ansatz_reps in ANSATZ_REPS_LIST:
                # Build model
                # model = qnn_torch_model # for purely quantum nn
                model = HybridModel(get_qnn_torch_model(entangle=entanglement,
                                                        feature_map_reps=feature_map_reps,
                                                        ansatz_reps=ansatz_reps)).to(device)  # Classical modifications in the HybridQNN class
                optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

                print(f"\n--- Starting Training {i}th---")
                train_losses = []
                test_losses = []

                for epoch in range(NUM_EPOCHS):
                    # Training phase
                    model.train()
                    running_loss = 0.0
                    for batch_X, batch_y in train_loader:
                        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                        optimizer.zero_grad()  # Clear gradients
                        outputs = model(batch_X)  # Forward pass
                        loss = LOSS(outputs, batch_y)  # Calculate loss
                        loss.backward()  # Backward pass (compute gradients)
                        optimizer.step()  # Update weights
                        running_loss += loss.item() * batch_X.size(0)

                    epoch_loss = running_loss / len(train_loader.dataset)
                    train_losses.append(epoch_loss)

                    # Validation/Test phase
                    model.eval()
                    test_loss = 0.0
                    with torch.no_grad():  # Disable gradient calculations
                        for batch_X_test, batch_y_test in test_loader:
                            batch_X_test, batch_y_test = batch_X_test.to(device), batch_y_test.to(device)
                            outputs_test = model(batch_X_test)
                            loss_test = LOSS(outputs_test, batch_y_test)
                            test_loss += loss_test.item() * batch_X_test.size(0)

                    epoch_test_loss = test_loss / len(test_loader.dataset)
                    test_losses.append(epoch_test_loss)

                    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Train Loss: {epoch_loss:.4f}, Test Loss: {epoch_test_loss:.4f}")

                print("--- Training Finished ---")

                # --- 9. Plotting Training History (Optional) ---
                # plt.figure(figsize=(10, 5))
                # plt.plot(train_losses, label='Training Loss')
                # plt.plot(test_losses, label='Test Loss')
                # plt.title('Training and Test Loss Over Epochs')
                # plt.xlabel('Epoch')
                # plt.ylabel('MSE Loss')
                # plt.legend()
                # plt.grid(True)
                # plt.show()

                # --- 10. Evaluation on Test (and Training) Set ---
                model.eval()
                all_preds = []
                all_targets = []
                all_preds_train = []
                all_targets_train = []
                with torch.no_grad():
                    for batch_X_test, batch_y_test in test_loader:
                        batch_X_test = batch_X_test.to(device)
                        outputs_test = model(batch_X_test)
                        all_preds.extend(outputs_test.cpu().numpy())
                        all_targets.extend(batch_y_test.cpu().numpy())
                    for batch_X_train, batch_y_train in train_loader:
                        batch_X_train = batch_X_train.to(device)
                        outputs_train = model(batch_X_train)
                        all_preds_train.extend(outputs_train.cpu().numpy())
                        all_targets_train.extend(batch_y_train.cpu().numpy())

                all_preds = np.array(all_preds)
                all_targets = np.array(all_targets)
                all_preds_temp = []
                for item in all_preds:
                    if item[1] > item[0]:
                        all_preds_temp.append(1)
                    else:
                        all_preds_temp.append(0)
                all_preds = np.array(all_preds_temp)

                all_preds_train = np.array(all_preds_train)
                all_targets_train = np.array(all_targets_train)
                all_preds_temp = []
                for item in all_preds_train:
                    if item[1] > item[0]:
                        all_preds_temp.append(1)
                    else:
                        all_preds_temp.append(0)
                all_preds_train = np.array(all_preds_temp)

                    # Example: Scatter plot for regression
                    # if NUM_TARGETS == 1: # Simple plot if single target variable
                    # plt.figure(figsize=(8, 8))
                    # plt.scatter(all_targets, all_preds, alpha=0.5)
                    # plt.plot([min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())],
                    #         [min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())],
                    #         'k--', lw=2, label='Ideal')
                    # plt.xlabel('Actual Values')
                    # plt.ylabel('Predicted Values')
                    # plt.title('Actual vs. Predicted Values on Test Set')
                    # plt.legend()
                    # plt.grid(True)
                    # plt.show()

                    # Further evaluation metrics can be added here (e.g., R-squared for regression, accuracy for classification)
                    
                print(torch.cuda.memory_allocated())
                print(torch.cuda.max_memory_allocated())

                print(f"\n--- Done for entanglement: {entanglement}, feature_map_reps: {feature_map_reps}, ansatz_reps: {ansatz_reps} ---")

                # Add to dataframe
                new_row = {'entanglement': entanglement,
                            'feature_map_reps': feature_map_reps,
                            'ansatz_reps': ansatz_reps,
                            'element test': element_test,
                            'actual test': np.array(all_targets).flatten(),
                            'predicted test': np.array(all_preds).flatten(),
                            'element train': element_train,
                            'actual train': np.array(all_targets_train).flatten(),
                            'predicted train': np.array(all_preds_train).flatten(),
                            }
                df.loc[len(df)] = new_row
                with np.printoptions(linewidth=10000):    # update csv every loop
                    df.to_csv(file_name, index=False)
    df.at[0, "info"] = [f"DATASET: {dataset_name}, LEARNING_RATE = {LEARNING_RATE}, "
                        f"BATCH_SIZE = {BATCH_SIZE}, NUM_EPOCHS = {NUM_EPOCHS}, LOSS: {LOSS}, "
                        f"CLASSIFIER_THRESHOLD = {CLASSIFIER_THRESHOLD}"]
    with np.printoptions(linewidth=10000):  # final csv output
        df.to_csv(file_name, index=False)


--- Start K-Fold Loop ---
Training data shape: X_train_t: torch.Size([20, 3]), y_train_t: torch.Size([20])
Testing data shape: X_test_t: torch.Size([1, 3]), y_test_t: torch.Size([1])
Assigned feature map parameters: 3
Assigned ansatz parameters: 6

--- Starting Training 0th---


/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.6937, Test Loss: 0.7031
Epoch 2/100, Train Loss: 0.6914, Test Loss: 0.7073
Epoch 3/100, Train Loss: 0.6890, Test Loss: 0.7114
Epoch 4/100, Train Loss: 0.6867, Test Loss: 0.7155
Epoch 5/100, Train Loss: 0.6844, Test Loss: 0.7194
Epoch 6/100, Train Loss: 0.6821, Test Loss: 0.7233
Epoch 7/100, Train Loss: 0.6797, Test Loss: 0.7270
Epoch 8/100, Train Loss: 0.6775, Test Loss: 0.7306
Epoch 9/100, Train Loss: 0.6752, Test Loss: 0.7341
Epoch 10/100, Train Loss: 0.6729, Test Loss: 0.7373
Epoch 11/100, Train Loss: 0.6707, Test Loss: 0.7404
Epoch 12/100, Train Loss: 0.6685, Test Loss: 0.7433
Epoch 13/100, Train Loss: 0.6663, Test Loss: 0.7461
Epoch 14/100, Train Loss: 0.6642, Test Loss: 0.7486
Epoch 15/100, Train Loss: 0.6621, Test Loss: 0.7509
Epoch 16/100, Train Loss: 0.6600, Test Loss: 0.7530
Epoch 17/100, Train Loss: 0.6579, Test Loss: 0.7549
Epoch 18/100, Train Loss: 0.6559, Test Loss: 0.7566
Epoch 19/100, Train Loss: 0.6539, Test Loss: 0.7580
Epoch 20/100, Train L

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.6940, Test Loss: 0.6868
Epoch 2/100, Train Loss: 0.6907, Test Loss: 0.6841
Epoch 3/100, Train Loss: 0.6874, Test Loss: 0.6814
Epoch 4/100, Train Loss: 0.6841, Test Loss: 0.6788
Epoch 5/100, Train Loss: 0.6807, Test Loss: 0.6762
Epoch 6/100, Train Loss: 0.6774, Test Loss: 0.6736
Epoch 7/100, Train Loss: 0.6740, Test Loss: 0.6711
Epoch 8/100, Train Loss: 0.6707, Test Loss: 0.6687
Epoch 9/100, Train Loss: 0.6673, Test Loss: 0.6663
Epoch 10/100, Train Loss: 0.6640, Test Loss: 0.6640
Epoch 11/100, Train Loss: 0.6606, Test Loss: 0.6617
Epoch 12/100, Train Loss: 0.6573, Test Loss: 0.6594
Epoch 13/100, Train Loss: 0.6541, Test Loss: 0.6569
Epoch 14/100, Train Loss: 0.6508, Test Loss: 0.6543
Epoch 15/100, Train Loss: 0.6476, Test Loss: 0.6516
Epoch 16/100, Train Loss: 0.6444, Test Loss: 0.6487
Epoch 17/100, Train Loss: 0.6413, Test Loss: 0.6457
Epoch 18/100, Train Loss: 0.6382, Test Loss: 0.6427
Epoch 19/100, Train Loss: 0.6351, Test Loss: 0.6395
Epoch 20/100, Train L

...


--- Starting Training 0th---
Epoch 1/100, Train Loss: 0.6934, Test Loss: 0.6919
Epoch 2/100, Train Loss: 0.6912, Test Loss: 0.6917
Epoch 3/100, Train Loss: 0.6891, Test Loss: 0.6917
Epoch 4/100, Train Loss: 0.6870, Test Loss: 0.6918
Epoch 5/100, Train Loss: 0.6848, Test Loss: 0.6921
Epoch 6/100, Train Loss: 0.6826, Test Loss: 0.6926
Epoch 7/100, Train Loss: 0.6805, Test Loss: 0.6932
Epoch 8/100, Train Loss: 0.6783, Test Loss: 0.6940
Epoch 9/100, Train Loss: 0.6761, Test Loss: 0.6949
Epoch 10/100, Train Loss: 0.6740, Test Loss: 0.6959
Epoch 11/100, Train Loss: 0.6718, Test Loss: 0.6970
Epoch 12/100, Train Loss: 0.6697, Test Loss: 0.6983
Epoch 13/100, Train Loss: 0.6676, Test Loss: 0.6996
Epoch 14/100, Train Loss: 0.6655, Test Loss: 0.7010
Epoch 15/100, Train Loss: 0.6634, Test Loss: 0.7024
Epoch 16/100, Train Loss: 0.6613, Test Loss: 0.7037
Epoch 17/100, Train Loss: 0.6593, Test Loss: 0.7051
Epoch 18/100, Train Loss: 0.6573, Test Loss: 0.7063
Epoch 19/100, Train Loss: 0.6553, Test Loss

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.6950, Test Loss: 0.6931
Epoch 2/100, Train Loss: 0.6915, Test Loss: 0.6954
Epoch 3/100, Train Loss: 0.6879, Test Loss: 0.6976
Epoch 4/100, Train Loss: 0.6843, Test Loss: 0.6997
Epoch 5/100, Train Loss: 0.6808, Test Loss: 0.7019
Epoch 6/100, Train Loss: 0.6772, Test Loss: 0.7041
Epoch 7/100, Train Loss: 0.6736, Test Loss: 0.7065
Epoch 8/100, Train Loss: 0.6700, Test Loss: 0.7090
Epoch 9/100, Train Loss: 0.6663, Test Loss: 0.7116
Epoch 10/100, Train Loss: 0.6627, Test Loss: 0.7143
Epoch 11/100, Train Loss: 0.6592, Test Loss: 0.7171
Epoch 12/100, Train Loss: 0.6556, Test Loss: 0.7198
Epoch 13/100, Train Loss: 0.6520, Test Loss: 0.7226
Epoch 14/100, Train Loss: 0.6485, Test Loss: 0.7252
Epoch 15/100, Train Loss: 0.6449, Test Loss: 0.7278
Epoch 16/100, Train Loss: 0.6414, Test Loss: 0.7305
Epoch 17/100, Train Loss: 0.6380, Test Loss: 0.7332
Epoch 18/100, Train Loss: 0.6345, Test Loss: 0.7361
Epoch 19/100, Train Loss: 0.6311, Test Loss: 0.7392
Epoch 20/100, Train L

/home/ashok/anaconda3/envs/qml46/lib/python3.11/site-packages/qiskit_machine_learning/connectors/torch_connector.py:306: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._weights.data = torch.tensor(initial_weights, dtype=torch.float)


Epoch 1/100, Train Loss: 0.6923, Test Loss: 0.6972
Epoch 2/100, Train Loss: 0.6912, Test Loss: 0.7001
Epoch 3/100, Train Loss: 0.6900, Test Loss: 0.7030
Epoch 4/100, Train Loss: 0.6889, Test Loss: 0.7059
Epoch 5/100, Train Loss: 0.6877, Test Loss: 0.7089
Epoch 6/100, Train Loss: 0.6866, Test Loss: 0.7118
Epoch 7/100, Train Loss: 0.6854, Test Loss: 0.7147
Epoch 8/100, Train Loss: 0.6843, Test Loss: 0.7176
Epoch 9/100, Train Loss: 0.6832, Test Loss: 0.7205
Epoch 10/100, Train Loss: 0.6821, Test Loss: 0.7233
Epoch 11/100, Train Loss: 0.6809, Test Loss: 0.7260
Epoch 12/100, Train Loss: 0.6798, Test Loss: 0.7286
Epoch 13/100, Train Loss: 0.6787, Test Loss: 0.7311
Epoch 14/100, Train Loss: 0.6776, Test Loss: 0.7334
Epoch 15/100, Train Loss: 0.6765, Test Loss: 0.7356
Epoch 16/100, Train Loss: 0.6754, Test Loss: 0.7376
Epoch 17/100, Train Loss: 0.6742, Test Loss: 0.7395
Epoch 18/100, Train Loss: 0.6731, Test Loss: 0.7412
Epoch 19/100, Train Loss: 0.6720, Test Loss: 0.7427
Epoch 20/100, Train L